<a href="https://colab.research.google.com/github/pastrop/kaggle/blob/master/lora_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== CELL 1: Install Dependencies =====
# !pip install -q transformers accelerate peft bitsandbytes datasets trl huggingface_hub
# from huggingface_hub import login
# login()


In [ ]:
# ===== CELL 2: Imports and Setup =====

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from trl import SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB"

In [ ]:
# ===== CELL 3: Load the Base Model with 4-bit Quantization =====

MODEL_ID = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (this takes ~1-2 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

print("Model loaded successfully!")
print(f"Model parameters: {model.num_parameters() / 1e9:.2f}B")

In [ ]:
# ===== CELL 4: Test the Base Model BEFORE Fine-Tuning =====

def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip()

test_prompts = [
    "Explain what LoRA adapters are in simple terms.",
    "What is context distillation and why is it useful?",
    "Write a short Python function that reverses a string.",
]

print("=" * 60)
print("BASE MODEL RESPONSES (before fine-tuning)")
print("=" * 60)
for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print(f"Response: {generate_response(model, tokenizer, prompt)}")
    print("-" * 40)

In [ ]:
# ===== CELL 5: Configure LoRA =====

lora_config = LoraConfig(
    r=8,                       # Rank — try 4, 8, 16, 32
    lora_alpha=16,             # Scaling factor (often 2x the rank)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "v_proj",
        # Uncomment to experiment with more layers:
        # "k_proj", "o_proj",
        # "gate_proj", "up_proj",
        # "down_proj",  # this is what the D2L paper targets
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ===== CELL 6: Prepare Training Data =====

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
#dataset = load_dataset("rajpurkar/squad", split="train")
dataset = dataset.shuffle(seed=42).select(range(2000))

print(f"Training samples: {len(dataset)}")
print(f"Categories: {set(dataset['category'])}")

def format_instruction(sample):
    if sample.get("context", "").strip():
        user_msg = f"{sample['instruction']}\n\nContext: {sample['context']}"
    else:
        user_msg = sample["instruction"]
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": sample["response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

print("\n--- Formatted training example ---")
print(format_instruction(dataset[0])[:500])


In [ ]:
# ===== CELL 7: Train with SFTTrainer =====

training_args = TrainingArguments(
    output_dir="./gemma2-lora-output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    formatting_func=format_instruction,
    max_seq_length=512,
    packing=True,
)

print("Starting LoRA fine-tuning (~15-30 min on T4)...")
trainer.train()
print("Training complete!")

In [ ]:
# ===== CELL 8: Test the Fine-Tuned Model =====

model.config.use_cache = True

print("=" * 60)
print("FINE-TUNED MODEL RESPONSES (after LoRA training)")
print("=" * 60)
for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print(f"Response: {generate_response(model, tokenizer, prompt)}")
    print("-" * 40)

In [ ]:
# ===== CELL 9: Save the LoRA Adapter =====

ADAPTER_PATH = "./gemma2-lora-adapter"
model.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")

# To reload later in a fresh session:
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, quantization_config=bnb_config, device_map="auto"
# )
# model_with_lora = PeftModel.from_pretrained(base_model, ADAPTER_PATH)


# ===== CELL 10: Context Distillation Demo =====
# This demonstrates the CD concept from the D2L paper (Equation 1).

print("\n" + "=" * 60)
print("EXPERIMENT: Simple Context Distillation")
print("=" * 60)

DOCUMENT = """
The Voyager 1 spacecraft was launched by NASA on September 5, 1977.
It is the most distant human-made object from Earth, currently over
15 billion miles away. Voyager 1 carries a Golden Record containing
sounds and images of Earth, intended as a message for any
extraterrestrial civilization that might find it. The spacecraft
entered interstellar space on August 25, 2012, making it the first
human-made object to do so. Its mission was originally to study
Jupiter and Saturn, but it has far exceeded its planned lifespan.
""".strip()

questions = [
    "When was Voyager 1 launched?",
    "What does Voyager 1 carry?",
    "When did Voyager 1 enter interstellar space?",
    "How far is Voyager 1 from Earth?",
    "What was Voyager 1's original mission?",
]

# Teacher: model WITH context
print("\n--- Teacher responses (WITH document in context) ---")
for q in questions:
    prompt_with_ctx = f"Based on this info:\n{DOCUMENT}\n\nQuestion: {q} Answer concisely."
    resp = generate_response(model, tokenizer, prompt_with_ctx, max_new_tokens=100)
    print(f"  Q: {q}")
    print(f"  A: {resp[:120]}")

# Student: model WITHOUT context
print("\n--- Student responses (WITHOUT document) ---")
for q in questions:
    prompt_no_ctx = f"Question: {q} Answer concisely."
    resp = generate_response(model, tokenizer, prompt_no_ctx, max_new_tokens=100)
    print(f"  Q: {q}")
    print(f"  A: {resp[:120]}")

print("""
The gap between teacher and student is exactly what Context
Distillation closes — and what D2L's hypernetwork learns to do
instantly in a single forward pass rather than via gradient descent.

NEXT STEPS:
  - Try different target_modules (especially "down_proj" as in D2L)
  - Experiment with LoRA ranks (4, 8, 16, 32) and compare quality
  - Implement full CD with KL divergence loss (Eq. 1 in the paper)
  - Explore the D2L codebase: github.com/SakanaAI/doc-to-lora
""")